# ML-KEM SampleNTT: software validation and mock traces

This supplemental notebook runs without capture hardware. It checks the
independent Python reference against the pinned C implementation, generates
synthetic traces, and exercises the storage and evaluation pipeline.

For your real **Husky + CW308_STM32F4** capture experiment, open
[the hardware lab](../Lab%207_1%20-%20ML-KEM%20SampleNTT%20Power%20Lab.ipynb).
The hardware notebook contains the explained connect, build, program, capture,
ADC tuning, batch, analysis, FLS and disconnect cells.

Synthetic results here do not establish measured leakage.


## 1. Use your notebook environment

Use the same Python kernel as the hardware lab. Install `requirements.txt`
from this folder if needed. A host `cc` compiler is needed for the independent
C comparison. This notebook never opens or programs a capture device.


In [1]:
from pathlib import Path
import sys, json, subprocess, hashlib
from dataclasses import asdict
from datetime import datetime, timezone
from uuid import uuid4

here = Path.cwd().resolve()
roots = [here, *list(here.parents)[:4]]
candidates = [p / 'custom/mlkem-samplentt-power-lab' for p in roots]
candidates += [p / 'courses/custom/mlkem-samplentt-power-lab' for p in roots]
candidates += [p / 'jupyter/courses/custom/mlkem-samplentt-power-lab' for p in roots]
# Also support a downloaded notebook with its companion lab beside it.
candidates += [p / 'mlkem-samplentt-power-lab' for p in roots]
if here.name == 'mlkem-samplentt-power-lab':
    candidates.insert(0, here)
LAB = next((p for p in candidates if (p / 'analysis/ground_truth.py').is_file()), None)
if LAB is None:
    raise FileNotFoundError('Cannot find jupyter/courses/custom/mlkem-samplentt-power-lab/. Set LAB to its absolute path if needed.')
sys.path.insert(0, str(LAB))
print('Kernel:', sys.executable)
print('Lab:', LAB)

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from analysis.ground_truth import ground_truth, generated_seed, output_digest
from analysis.c_reference import verify_sources, build_reference, CReference
from capture.records import acquire_batch, save_dataset, load_dataset
from simulation.mock import MockBackend, MockSettings
from analysis.evaluate import evaluate_mock, mock_seed_segments
from analysis.plots import plot_mock_summary

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
RUN = LAB / 'results/runs' / RUN_ID
RUN.mkdir(parents=True)
print('This run:', RUN)

Kernel: /home/ubuntu/Desktop/work/chipwhisperer-new/.venv/bin/python
Lab: /home/ubuntu/Desktop/work/chipwhisperer-new/jupyter/courses/custom/mlkem-samplentt-power-lab
This run: /home/ubuntu/Desktop/work/chipwhisperer-new/jupyter/courses/custom/mlkem-samplentt-power-lab/results/runs/20261029T004034Z-61c4f086


## 2. Inspect the real implementations

We use the **top-level** `algorithmA0` / `algorithmA1` implementation at ProjectTempo commit `fd72834bd789c644b52fe0d62b2fa3b9f34dfaec`. See `docs/reference-inspection.md` for the source inventory and dependencies.

| Variant | Actual function | Source |
|---|---|---|
| Variable time | `void algorithmA0(int16_t a[256], const uint8_t extseed[34])` | `src/algorithm_a0.c` |
| FLS | `void algorithmA1(int16_t a[256], const uint8_t extseed[34])` | `src/algorithm_a1.c` |

Two experimental details differ from simplified descriptions:

- A0 stops after **at most 280 pairs**, even if the output has not filled. A shortfall is a failure, not an implicit fallback.
- The top-level A1 makes a tentative write for rejected candidates at the current output slot. Its counter can advance beyond 256. The pqm4 copy has different masking and counter behavior. Do not merge their power results.

FLS executes **280 pairs / 560 comparisons**. After 256 valid candidates fill the output, candidate validity continues to vary, but no new output coefficient is committed. Fixed source loops are not proof of constant-time compiled execution.

In [2]:
manifest = verify_sources()
print('Pinned repository:', manifest['repository'])
print('Pinned commit:', manifest['commit'])
for variant in ('a0', 'a1'):
    path = LAB / f'reference/tempo/src/algorithm_{variant}.c'
    print(f'\n--- {path.name} ---\n{path.read_text()}')

Pinned repository: https://github.com/afonsoarriaga/ProjectTempo
Pinned commit: fd72834bd789c644b52fe0d62b2fa3b9f34dfaec

--- algorithm_a0.c ---
/********************************************************************************
 * Project: Tempo                                                               *
 *                                                                              *
 * Name: Algorithm A0                                                           *
 * Description: VT rejection sampling [Algorithm 7, FIPS 203]                   *
 ********************************************************************************/

#include "algorithms.h"
#include "config.h"
#include "fips202.h"

void algorithmA0(int16_t a[KYBER_N], const uint8_t extseed[KYBER_SYMBYTES + 2]) {
    uint8_t C[3];
    unsigned int j = 0, iter = 0;

    shake128incctx state;
    shake128_inc_init(&state);
    shake128_inc_absorb(&state, extseed, KYBER_SYMBYTES + 2);
    shake128_inc_finalize(&state);
    
 

## 3. Validate the independent Python ground truth

The Python reference obtains one SHAKE128 prefix with `hashlib`, then decodes two 12-bit candidates from each three-byte pair. It does not call the C implementation to obtain candidates or labels.

Input order is **ρ || x || y**. For an untransposed matrix entry `A[row, column]`, use `x=column`, `y=row`. The defaults below choose one polynomial.

We compare all 256 coefficients against unchanged C, check committed regression vectors, test byte-order and comparison boundaries, and exercise shortfall/final-pair cases. The regression vectors are laboratory C-generated vectors, not official NIST KATs.

In [ ]:
test = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=LAB,
                      text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test.stdout)
if test.returncode:
    raise RuntimeError('Correctness tests failed. Resolve before acquisition.')

reference = CReference(build_reference(RUN / 'host-build'))
for number in range(16):
    seed = generated_seed(number, 'notebook-validation-v1')
    for variant in ('variable_time', 'fls'):
        gt = ground_truth(seed, 1, 2, variant)
        assert gt.complete
        assert gt.coefficients == reference.sample(seed, 1, 2, variant)
print('Additional C/Python checks: 16 seed/index pairs × both variants passed.')

In [ ]:
SEED = bytes(range(32))
X_INDEX, Y_INDEX = 1, 2
truth = {v: ground_truth(SEED, X_INDEX, Y_INDEX, v)
         for v in ('variable_time', 'fls')}
assert truth['variable_time'].coefficients == truth['fls'].coefficients
rows = ['| Variant | Pair iterations | Decoded candidates | Valid | Committed | SHAKE rate blocks |',
        '|---|---:|---:|---:|---:|---:|']
for name, gt in truth.items():
    rows.append(f'| {name} | {gt.pair_iterations} | {len(gt.candidates)} | {sum(gt.valid)} | {sum(gt.committed)} | {gt.xof_rate_blocks} |')
display(Markdown('\n'.join(rows)))
print('First 16 coefficients:', truth['fls'].coefficients[:16])
print('Verification SHA3-256(LE16 coefficients):', output_digest(truth['fls'].coefficients).hex())
(RUN / 'example-ground-truth.json').write_text(json.dumps({v:g.to_dict() for v,g in truth.items()}, indent=2) + '\n')

In [ ]:
fls = truth['fls']
fig, axes = plt.subplots(2, 1, figsize=(11, 5), constrained_layout=True)
k = np.arange(len(fls.candidates))
axes[0].scatter(k, fls.candidates, c=np.where(fls.valid, '#19758c', '#d67b35'), s=8)
axes[0].axhline(3329, color='#a13b57', ls='--', label='Acceptance boundary: candidate < 3329')
axes[0].set(ylabel='12-bit candidate', title='Computed ground truth — no measured power')
axes[0].legend()
axes[1].plot(k, np.cumsum(fls.committed), color='#376c62', label='Committed coefficients')
axes[1].plot(k, np.cumsum(fls.valid), color='#a14c79', label='Valid candidates encountered')
axes[1].axvline(fls.candidates_to_fill - 1, color='0.5', ls=':', label='Output first fills')
axes[1].set(xlabel='Candidate index', ylabel='Cumulative count')
axes[1].legend()
fig.savefig(RUN / 'ground-truth.png', dpi=150)
plt.show()

## 4. Acquire synthetic traces

This mock backend **injects** candidate-validity and Hamming-weight activity plus Gaussian noise. It also injects variable A0 duration and fixed FLS duration with small measurement jitter. It is an invented leakage model, not a Cortex-M or analog simulator.

Use this section to validate storage, plotting, input grouping and evaluation. Change the strengths to zero to remove the injected signal; change `noise_sd` to explore noise. These experiments cannot establish whether FLS leaks on real hardware.

The same 96 generated laboratory seeds are used for both variants. Two repetitions are captured in randomized order. Every capture saves the seed, literal input indices, candidates, validity/commit labels, waveform and provenance.

In [ ]:
N_SEEDS, REPEATS = 96, 2
settings = MockSettings(rng_seed=20261028, noise_sd=0.85,
                        variable_time_strength=1.2, fls_strength=0.55)
backend = MockBackend(settings)
seeds = [generated_seed(i) for i in range(N_SEEDS)]
datasets = {}
for variant in ('variable_time', 'fls'):
    records, failures = acquire_batch(backend, seeds, variant, repeats=REPEATS,
                                      x=X_INDEX, y=Y_INDEX, order_seed=42)
    path = save_dataset(RUN / variant, records, failures,
                        {'mock_settings': asdict(settings), 'n_seeds': N_SEEDS, 'repeats': REPEATS})
    restored, saved = load_dataset(path)
    assert len(restored) == len(records)
    assert not failures, failures
    datasets[variant] = restored
    print(variant, len(records), 'synthetic captures; failures:', len(failures))

In [ ]:
fig = plot_mock_summary(datasets)
fig.savefig(RUN / 'synthetic-summary.png', dpi=160)
plt.show()

## 5. Check an acceptance classifier on held-out seeds

Candidate **0** is selected in advance. For each seed, repetitions of its segment are averaged. The train/test split uses distinct seeds, and scaling is fitted only on training data. Compare single and repeated traces with the same split.

A random candidate is valid with probability `3329/4096 ≈ 0.813`, so a majority predictor can have high raw accuracy. We report balanced accuracy, precision/recall for the valid class, ROC-AUC, and a confusion matrix with labels `[rejected, valid]`. The majority baseline has balanced accuracy 0.5.

The descriptive Welch test uses one mean amplitude per **training seed** in a preselected segment. It is not a full TVLA campaign, and its p-value does not account for exploratory window selection or multiple testing. Hardware traces need measured candidate boundaries; this mock segmenter explicitly refuses them.

In [ ]:
metrics = {}
rows = ['| Synthetic model | Repeats | Test seeds | Balanced accuracy | Precision valid | Recall valid | ROC-AUC |',
        '|---|---:|---:|---:|---:|---:|---:|']
for variant, records in datasets.items():
    metrics[variant] = {}
    for repeats in (1, REPEATS):
        subset = [r for r in records if r.metadata['repeat'] < repeats]
        result = evaluate_mock(subset, candidate_index=0, split_seed=123)
        assert not set(result['train_seeds']) & set(result['test_seeds'])
        metrics[variant][str(repeats)] = result
        rows.append(f"| {variant} | {repeats} | {result['independent_test_seeds']} | {result['balanced_accuracy']:.3f} | {result['precision_valid']:.3f} | {result['recall_valid']:.3f} | {result['roc_auc']:.3f} |")
display(Markdown('\n'.join(rows)))
for variant in datasets:
    result = metrics[variant][str(REPEATS)]
    print(variant, 'confusion matrix, rows=true, columns=predicted, [reject, valid]:')
    print(np.array(result['confusion_matrix_reject_valid']))
    print('Descriptive training-seed Welch t:', round(result['descriptive_train_welch_t'], 3))
(RUN / 'synthetic-metrics.json').write_text(json.dumps(metrics, indent=2, allow_nan=False) + '\n')
print('All results above are SYNTHETIC.')

## 6. Compare the injected timing and power signals

This verifies that the analysis software can represent a scenario with fixed timing and varying power. The model was explicitly constructed to behave this way. A real Level 3 countermeasure evaluation requires measurements, an assembly review, and a separate known-candidate-seed experiment on unseen seed sets.

Pair iteration count is not a complete timing model: SHAKE blocks, allocation, branches and the full fixed scans also contribute on a real MCU.

In [ ]:
for variant, records in datasets.items():
    timing = np.array([r.metadata['timing_value'] for r in records])
    count = np.array([r.truth['pair_iterations'] for r in records])
    correlation = float(np.corrcoef(timing, count)[0, 1]) if np.std(count) > 0 else None
    print(f'{variant}: synthetic timing mean={timing.mean():.2f}, SD={timing.std(ddof=1):.2f}; pair-count correlation={correlation}')
print('No hardware leakage detection or candidate-seed distinction claim follows from this simulation.')

## Continue on hardware

The software pipeline can be tested here, including varying mock noise and
leakage parameters. Hardware capture is in the main **Lab 7_1** notebook in
`jupyter/courses/custom`, configured for Husky + CW308_STM32F4.
